# ADVC — Colab Setup & Phase 1 Evaluation
Run cells **top to bottom** on a fresh Colab session.

> **Before starting:** Make sure GPU is enabled — Runtime → Change runtime type → T4 GPU

## 1 — Check GPU

In [ ]:
!nvidia-smi

## 2 — Clone repo & navigate

In [ ]:
%cd /content
!git clone https://github.com/Jmanav/ADVC.git ADVC-main
%cd ADVC-main

## 3 — Install dependencies

In [ ]:
!pip install -r requirements.txt

## 4 — Verify PyTorch + GPU

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
print('GPU            :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('Torch version  :', torch.__version__)

## 5 — Kaggle setup
**Before running this cell:**
1. Go to `https://www.kaggle.com` → profile → Settings → API → **Create New Token** — downloads `kaggle.json`
2. Accept competition terms at `https://www.kaggle.com/competitions/imagenet-object-localization-challenge` → **Join Competition**

In [ ]:
from google.colab import files
import os, pathlib

uploaded = files.upload()  # select kaggle.json from your machine

kaggle_dir = pathlib.Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
os.rename('kaggle.json', kaggle_dir / 'kaggle.json')
os.chmod(kaggle_dir / 'kaggle.json', 0o600)
print('kaggle.json placed at', kaggle_dir / 'kaggle.json')

## 6 — Download 1000 ImageNet val images

The competition stores images as individual files — there is no single val tar to download.
This cell:
1. Downloads `LOC_val_solution.csv` (~2 MB) to get image→class label mapping
2. Picks 1000 images deterministically (seed=42)
3. Downloads only those images directly into `data/imagenet/val/<synset>/`

Total download: ~150 MB, takes ~10–15 minutes.

In [ ]:
import csv, os, pathlib, random
!pip install -q --upgrade kaggle
from kaggle.api.kaggle_api_extended import KaggleApiExtended

COMP   = 'imagenet-object-localization-challenge'
SEED   = 42
N      = 1000
VAL_DIR = 'data/imagenet/val'

api = KaggleApiExtended()
api.authenticate()

# ── Step 1: download the label CSV (~2 MB) ────────────────────────────────────
os.makedirs('data/imagenet', exist_ok=True)
print('Downloading LOC_val_solution.csv ...')
api.competition_download_file(COMP, 'LOC_val_solution.csv', path='data/imagenet')

# ── Step 2: parse image_id → synset ──────────────────────────────────────────
labels = {}
with open('data/imagenet/LOC_val_solution.csv') as f:
    for row in csv.DictReader(f):
        img_id = row['ImageId']                    # e.g. ILSVRC2012_val_00000001
        synset = row['PredictionString'].split()[0] # first token is the synset
        labels[img_id] = synset
print(f'Label file loaded — {len(labels)} val images')

# ── Step 3: deterministic 1000-image subset (seed=42) ─────────────────────────
random.seed(SEED)
selected = random.sample(sorted(labels.keys()), N)

# ── Step 4: skip already-downloaded images ────────────────────────────────────
to_download = []
for img_id in selected:
    synset = labels[img_id]
    dest   = pathlib.Path(VAL_DIR) / synset / f'{img_id}.JPEG'
    if not dest.exists():
        to_download.append(img_id)

print(f'{N - len(to_download)} already present, downloading {len(to_download)} images ...')

# ── Step 5: download one by one ───────────────────────────────────────────────
for i, img_id in enumerate(to_download):
    synset      = labels[img_id]
    remote_path = f'ILSVRC/Data/CLS-LOC/val/{img_id}.JPEG'
    dest_dir    = pathlib.Path(VAL_DIR) / synset
    dest_dir.mkdir(parents=True, exist_ok=True)

    try:
        api.competition_download_file(COMP, remote_path, path=str(dest_dir))
    except Exception as e:
        print(f'  WARN: {img_id} failed — {e}')
        continue

    if (i + 1) % 100 == 0:
        print(f'  {i + 1}/{len(to_download)} downloaded')

print(f'\nDone — images in {VAL_DIR}')

## 7 — Verify ImageFolder

In [ ]:
from torchvision.datasets import ImageFolder

ds = ImageFolder('data/imagenet/val')
print(f'Images  : {len(ds)}')
print(f'Classes : {len(ds.classes)}')
print(f'Sample  : {ds.classes[:5]} ...')

if len(ds) < 900:
    print('WARNING: fewer than 900 images — some downloads may have failed. Re-run Cell 6.')
else:
    print('OK — ready to run experiments.')

## 8 — Run Phase 1 evaluation
Loops over `fp32 → int8 → int4` × `fgsm / pgd / patch`.  
Results are saved to `results/phase1_results.csv` after each combination.  
Safe to interrupt and re-run — completed rows are skipped automatically.

In [ ]:
!python experiments/eval_phase1.py --model deit_small

## 9 — View results

In [ ]:
import pandas as pd

df = pd.read_csv('results/phase1_results.csv')
df['clean_acc']       = df['clean_acc'].map('{:.4f}'.format)
df['robust_acc']      = df['robust_acc'].map('{:.4f}'.format)
df['asr']             = df['asr'].map('{:.4f}'.format)
df['robustness_gap']  = df['robustness_gap'].map('{:.4f}'.format)
df

## 10 — (Optional) Save results to Google Drive
Colab wipes `/content` on runtime reset. Mount Drive to persist results.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

dest = '/content/drive/MyDrive/ADVC/results'
shutil.copytree('results', dest, dirs_exist_ok=True)
print(f'Results saved to {dest}')